# Revisión y validación del matching YouTube ↔ Local

Objetivo: revisar los 62 matches automáticos, corregir los falsos positivos,
y recuperar posibles matches entre los 5 SIN MATCH.

Flujo:
1. Ver la tabla de matches OK ordenada por score (peores primero)
2. Editar el dict `corrections` con lo que querés cambiar
3. Ver los 5 videos YouTube sin match y rescatar manualmente si corresponde
4. Guardar el CSV corregido

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

CSV_PATH     = Path('../data/catalog/yt_vs_local_matching.csv')
OCR_JSON     = Path('../data/catalog/yt_ocr.json')

df  = pd.read_csv(CSV_PATH)
ok  = df[df['match'] == 'OK'].copy().sort_values('score')      # peores primero
sin = df[df['match'] == 'SIN MATCH'].copy().sort_values('yt_dur_s')

print(f'Matches OK:   {len(ok)}')
print(f'Sin match:    {len(sin)}')
print(f'Total videos: {len(df)}')

## 2. Matches OK — peores primero

Revisá especialmente los de score < 0.55 (fila superior de la tabla).

In [ ]:
cols = ['score', 'tfidf_score', 'dur_score', 'yt_title', 'local_file',
        'yt_dur_s', 'local_dur_s', 'yt_topic', 'local_topic', 'has_sub']

display(
    ok[cols]
    .style
    .background_gradient(subset=['score'],       cmap='RdYlGn', vmin=0.25, vmax=0.75)
    .background_gradient(subset=['tfidf_score'], cmap='Blues',  vmin=0.0,  vmax=0.5)
    .background_gradient(subset=['dur_score'],   cmap='Greens', vmin=0.6,  vmax=1.0)
    .format({
        'score':       '{:.3f}',
        'tfidf_score': '{:.3f}',
        'dur_score':   '{:.3f}',
        'yt_dur_s':    '{:.0f}s',
        'local_dur_s': '{:.1f}s',
    })
    .set_properties(**{'text-align': 'left', 'font-size': '12px'})
)

In [ ]:
import sys
sys.path.insert(0, '..')
from utils.registry import load_registry, save_registry, update_entry

reg = load_registry()
df_reg = pd.DataFrame(reg).T
df_reg

In [11]:
update_entry(reg, "DSC_0785 Retiro si", review_status="ok")
save_registry(reg)

[registry] Guardado: /home/mdipaola/lsa-dataset-toolkit/notebooks/../data/catalog/video_registry.json  (41 videos)


## 3. Correcciones manuales

Editá el dict según lo que encontraste en la tabla:
- `"RECHAZAR"` → el match es incorrecto, no hay video local para este YT
- `"DSC_XXXX nombre.MOV"` → reemplazar por el archivo local correcto
- Para agregar un SIN MATCH recuperado: usá `rescued` más abajo

In [ ]:
# ── Editá este dict ────────────────────────────────────────────────────────────
corrections = {

    # "hHqyvAOZpwk": "RECHAZAR",
    "O_9wfdgHaW4": "DSC_0773 Pension fliar si.MOV",
}

# Videos SIN MATCH que querés rescatar manualmente:
rescued = {
    # "yt_id": "DSC_XXXX nombre.MOV",
}
# ──────────────────────────────────────────────────────────────────────────────

print(f'Correcciones: {len(corrections)}  |  Rescatados: {len(rescued)}')

In [ ]:
df_edit = df.copy()

for yt_id, val in corrections.items():
    if val == 'RECHAZAR':
        df_edit.loc[df_edit['yt_id'] == yt_id, ['match', 'local_file']] = ['RECHAZADO', None]
        print(f'  RECHAZADO : {yt_id}')
    else:
        df_edit.loc[df_edit['yt_id'] == yt_id, 'local_file'] = val
        print(f'  CORREGIDO : {yt_id}  →  {val}')

for yt_id, local_file in rescued.items():
    mask = df_edit['yt_id'] == yt_id
    df_edit.loc[mask, ['match', 'local_file']] = ['OK', local_file]
    print(f'  RESCATADO : {yt_id}  →  {local_file}')

print(f"\nMatches OK tras correcciones: {(df_edit['match'] == 'OK').sum()}")

## 4. Videos SIN MATCH

Lista de los 26 videos YouTube sin match local. Si reconocés alguno, agregalo al dict `rescued` arriba.

## 3. Videos SIN MATCH

Los 5 videos YouTube sin match. Si reconocés alguno, agregalo al dict `rescued` en la celda de correcciones.

In [ ]:
# Videos YouTube sin match local
print('=== YouTube SIN MATCH ===')
display(
    sin[['yt_id', 'yt_title', 'yt_dur_s', 'yt_topic', 'playlist']]
    .style
    .set_properties(**{'text-align': 'left', 'font-size': '12px'})
    .format({'yt_dur_s': '{:.0f}s'})
)

# MOV locales sin match: los que no aparecen como local_file en ningún match OK
matched_files = set(df[df['match'] == 'OK']['local_file'].dropna())
all_files_seen = set(df['local_file'].dropna())
unmatched_movs = sorted(all_files_seen - matched_files)

print(f'\n=== MOV locales sin match ({len(unmatched_movs)}) ===')
for f in unmatched_movs:
    print(f'  {f}')

## 6. Validación con OCR (Paso 3a)

Después de correr `scripts/fetch_subs.py`, esta celda carga el reporte OCR
y cruza la sincronización sub↔video como evidencia adicional del match.

> Si `sync_ok=False` el sub del YouTube no encaja con la duración del MOV local
> — posible match incorrecto o offset de tiempo.

In [ ]:
if OCR_REPORT.exists():
    ocr = pd.read_csv(OCR_REPORT)
    ok_now = df_edit[df_edit['match'] == 'OK'].copy()
    merged = ok_now.merge(ocr[['yt_id', 'n_segments', 'ocr_confidence',
                                'docx_match_ratio', 'sync_ok']], on='yt_id', how='left')

    suspect = merged[merged['sync_ok'] == False]
    good    = merged[merged['sync_ok'] == True]
    missing = merged[merged['sync_ok'].isna()]

    print(f'sync_ok=True:  {len(good):2d}  — match confirmado por OCR')
    print(f'sync_ok=False: {len(suspect):2d}  — revisar')
    print(f'sin datos OCR: {len(missing):2d}')

    if len(suspect):
        print('\n⚠ Sospechosos (sync_ok=False):')
        display(suspect[['yt_title', 'local_file', 'score',
                          'sync_ok', 'n_segments', 'ocr_confidence']]
                .style.applymap(lambda v: 'background-color: #ffcccc' if v is False else '',
                                subset=['sync_ok']))
else:
    print('⏳ OCR report no disponible aún — correr scripts/fetch_subs.py primero')
    print(f'   (buscando: {OCR_REPORT})')

## 7. Guardar CSV corregido

In [ ]:
# Resumen antes de guardar
counts = df_edit['match'].value_counts()
for k, v in counts.items():
    print(f'  {k:12s}: {v}')
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ok_final = df_edit[df_edit['match'] == 'OK']
if len(ok_final):
    axes[0].hist(ok_final['score'], bins=15, color='#2ecc71', edgecolor='white')
    axes[0].axvline(0.35, color='red', linestyle='--', label='Threshold original 0.35')
    axes[0].set_xlabel('Score de matching')
    axes[0].set_ylabel('Videos')
    axes[0].set_title('Distribución de scores (matches OK)')
    axes[0].legend()

cats   = ['OK', 'SIN MATCH', 'RECHAZADO']
vals   = [counts.get(c, 0) for c in cats]
colors = ['#2ecc71', '#e74c3c', '#f39c12']
axes[1].bar(cats, vals, color=colors, edgecolor='white')
axes[1].set_title('Estado del matching')
for i, v in enumerate(vals):
    if v: axes[1].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
df_edit.to_csv(CSV_PATH, index=False, encoding='utf-8')
print(f'Guardado: {CSV_PATH}')
print(f'Matches OK finales: {(df_edit["match"] == "OK").sum()}')